# Credit card fraud detector

In this solution we will build the core of a credit card fraud detection system locally. We will start by training an anomaly detection algorithm, then proceed to train two XGBoost models for supervised training. To deal with the highly unbalanced data common in fraud detection, our first model will use re-weighting of the data, and the second will use re-sampling, using the popular SMOTE technique for oversampling the rare fraud data.

Our solution includes an example of making calls to a REST API to simulate a real deployment, using ```FastAPI``` to trigger both the anomaly detection and XGBoost model.

## Set up environment

In [ ]:
import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import IsolationForest
import xgboost as xgb
import boto3
import joblib
from fraudwatcher.definitions import *
from dotenv import load_dotenv
load_dotenv()

## Set up environnement

In [ ]:
# Configuration des variables d'environnement
aws_region = os.environ.get('AWS_REGION')
aws_access_key = os.getenv("AWS_ID_ACCESS_KEY")
aws_secret_key = os.getenv("AWS_SECRET_ACCESS_KEY")
s3_bucket = os.getenv("SOLUTIONS_S3_BUCKET")
s3_prefix = os.getenv("SOLUTION_NAME")

print(f"aws_region: {aws_region}")
print(f"aws_access_key: {aws_access_key}")
print(f"aws_secret_key: {aws_secret_key}")
print(f"s3_bucket: {s3_bucket}")
print(f"s3_prefix: {s3_prefix}")

In [ ]:
os.makedirs(DATASET_PATH, exist_ok=True)
os.makedirs(CHECKPOINTS_PATH, exist_ok=True)

In [ ]:
# Initialisation du client S3
s3_client = boto3.client(
    's3',
    aws_access_key_id=aws_access_key,
    aws_secret_access_key=aws_secret_key,
    region_name=aws_region
)

In [ ]:
# Download file from S3
s3_key = f"{s3_prefix}/creditcard.csv.zip"
local_zip_path = f"{DATASET_PATH}/creditcard.csv.zip"

print("Téléchargement en cours...")
s3_client.download_file(s3_bucket, s3_key, local_zip_path)
print(f"Téléchargement terminé : {local_zip_path}")

In [ ]:
# Unzip file to DATASET_PATH
print("Décompression...")
with zipfile.ZipFile(local_zip_path, 'r') as zip_ref:
    zip_ref.extractall(DATASET_PATH)
print(f"Fichiers extraits dans le dossier '{DATASET_PATH}'.")

In [ ]:
# (Optionnal) Remove zip file
os.remove(local_zip_path)

## Investigate and process the data

Let's start by reading in the credit card fraud data set.

In [ ]:
data = pd.read_csv(f"{DATASET_PATH}/creditcard.csv", delimiter=',')
data.head()

Let's take a peek at our data (we only show a subset of the columns in the table):

In [ ]:
print(data.columns)
data[['Time', 'V1', 'V2', 'V27', 'V28', 'Amount', 'Class']].describe()

The dataset contains only numerical features, because the original features have been transformed using PCA, to protect user privacy. As a result, the dataset contains 28 PCA components, V1-V28, and two features that haven't been transformed, Amount and Time. Amount refers to the transaction amount, and Time is the seconds elapsed between any transaction in the data and the first transaction.

The class column corresponds to whether or not a transaction is fraudulent. We see that the majority of data is non-fraudulent with only 
 (
) of the data corresponding to fraudulent examples, out of the total of 284,807 examples in the data.

In [ ]:
nonfrauds, frauds = data.groupby('Class').size()
print('Number of transactions: ', nonfrauds + frauds)
print('Number of frauds: ', frauds)
print('Number of non-frauds: ', nonfrauds)
print('Percentage of fradulent data:', 100.*frauds/(frauds + nonfrauds))

We already know that the columns $V_i$ have been normalized to have mean and unit standard deviation as the result of a PCA.

In [ ]:
feature_columns = data.columns[:-1]
label_column = data.columns[-1]

features = data[feature_columns].values.astype('float32')
labels = (data[label_column].values).astype('float32')

features.shape, labels.shape

Next, we will prepare our data for loading and training.

## Training

We will split our dataset into a train and test to evaluate the performance of our models. It's important to do so before any techniques meant to alleviate the class imbalance are used. This ensures that we don't leak information from the test set into the train set.

In [ ]:
# train set and test set
X, X_test, y, y_test = train_test_split(features, labels, test_size=0.1, random_state=42, stratify=labels)
# validation set
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, random_state=42, stratify=y)

print('Training data X: ', X_train.shape, ' y: ', y_train.shape)
print('Validation data X: ', X_val.shape, ' y: ', y_val.shape)
print('Test data X: ', X_test.shape, ' y: ', y_test.shape)


In [ ]:
np.unique(y_train, return_counts=True), np.unique(y_test, return_counts=True)

> Note: If you are bringing your own data to this solution and they include categorical data, that have strings as values, you'd need to one-hot encode these values first using for example sklearn's OneHotEncoder, as XGBoost only supports numerical data.

### Unsupervised Learning (Anomaly detection)

In a fraud detection scenario, commonly we will have very few labeled examples, and it's possible that labeling fraud takes a very long time. We would like then to extract information from the unlabeled data we have at hand as well. Anomaly detection is a form of unsupervised learning where we try to identify anomalous examples based solely on their feature characteristics. Isolation-Forest is a state-of-the-art anomaly detection algorithm that is both accurate and scalable. We will train such a model on our training data and evaluate its performance on our test set.

In [ ]:
# import matplotlib.pyplot as plt

# scatter = plt.scatter(X_train[:, 1], X_train[:, 14], c=y_train, s=15, edgecolor="k")
# handles, labels = scatter.legend_elements()
# plt.axis("square")
# plt.legend(handles=handles, labels=["inliers", "outliers"], title="true class")
# plt.title("Gaussian inliers with \nuniformly distributed outliers")
# plt.show()

### Isolation Forest

In [ ]:
clf = IsolationForest(max_samples=100, random_state=0)
clf.fit(X_train)

In [ ]:
np.unique(clf.predict(X_test), return_counts=True)

In [ ]:
positives = X_test[y_test == 1] # frauds
positives_scores = clf.decision_function(positives)

negatives = X_test[y_test == 0]
negatives_scores = clf.decision_function(negatives)

In [ ]:
positives_scores

In [ ]:
negatives_scores

In [ ]:
sns.set(color_codes=True)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 15))
sns.histplot(positives_scores, label='fraud', bins=7, ax=ax)
sns.histplot(negatives_scores, label='not-fraud', bins=100, ax=ax)
ax.legend()

In [ ]:
# Save model
model_path = f"{CHECKPOINTS_PATH}/isoforest-model.joblib"
joblib.dump(clf, model_path)
print(f"Modèle sauvegardé dans '{model_path}'.")
print("Terminé")

In [ ]:
import matplotlib.pyplot as plt

from sklearn.dummy import DummyClassifier
from sklearn.metrics import DetCurveDisplay, RocCurveDisplay

fig, [ax_roc, ax_det] = plt.subplots(1, 2, figsize=(11, 5))

ax_roc.set_title("Receiver Operating Characteristic (ROC) curves")
ax_det.set_title("Detection Error Tradeoff (DET) curves")

ax_roc.grid(linestyle="--")
ax_det.grid(linestyle="--")

for name, clf in classifiers.items():
    (color, linestyle) = (
        ("black", "--") if name == "Non-informative baseline" else (None, None)
    )
    clf.fit(X_train, y_train)
    RocCurveDisplay.from_estimator(
        clf,
        X_test,
        y_test,
        ax=ax_roc,
        name=name,
        curve_kwargs=dict(color=color, linestyle=linestyle),
    )
    DetCurveDisplay.from_estimator(
        clf, X_test, y_test, ax=ax_det, name=name, color=color, linestyle=linestyle
    )

plt.legend()
plt.show()